# Prepare shared Causal Chambers wind-tunnel measurements data

This notebook creates or verifies the frozen `causalChamber` bundles used by
all seven method notebooks in this directory. Data generation and acquisition
are kept separate from model fitting so every method reads byte-identical
observations and the same optional contextual reference network.

This case study uses measured wind-tunnel time series from the Causal Chambers project. The supplied physical causal model is retained as a separately labelled reference panel; the empirical comparison does not require recovery metrics.

In [ ]:
from __future__ import annotations

import hashlib
import importlib.util
import json
import math
import os
import platform
import subprocess
import sys
import tempfile
import types
import urllib.request
import zipfile
import pickle
from pathlib import Path

EXPECTED_PYTHON = (3, 14)
if sys.version_info[:2] != EXPECTED_PYTHON:
    raise RuntimeError(
        "This notebook requires CPython 3.14; "
        f"found {platform.python_version()}."
    )

import numpy as np
import pandas as pd
import scipy.special

PROJECT_ROOT = next(
    (
        path
        for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (path / "src" / "fastCGC" / "__init__.py").exists()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the fastC-GC project root.")
SOURCE_ROOT = PROJECT_ROOT / "src"
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))

from fastCGC.experiments.upstream import verify_upstream_revision

In [ ]:
def series_digest(series):
    return hashlib.sha256(
        np.ascontiguousarray(series, dtype=np.float64).tobytes()
    ).hexdigest()


def save_bundle(
    path,
    *,
    series,
    metadata,
    aggregate_truth=None,
    lag_truth=None,
    reference_graph=None,
    extra_arrays=None,
):
    graph_arrays = (aggregate_truth, lag_truth, reference_graph)
    if sum(value is not None for value in graph_arrays) > 1:
        raise ValueError("Provide at most one truth or reference graph.")
    path.parent.mkdir(parents=True, exist_ok=True)
    arrays = {"series": np.asarray(series, dtype=float)}
    if lag_truth is not None:
        arrays["lag_truth"] = np.asarray(lag_truth, dtype=bool)
    elif aggregate_truth is not None:
        arrays["aggregate_truth"] = np.asarray(
            aggregate_truth,
            dtype=bool,
        )
    elif reference_graph is not None:
        arrays["reference_graph"] = np.asarray(
            reference_graph,
            dtype=bool,
        )
    for name, value in (extra_arrays or {}).items():
        if name in arrays:
            raise ValueError(f"Duplicate bundle array name: {name}.")
        arrays[name] = np.asarray(value)
    digest = series_digest(arrays["series"])
    metadata = {**metadata, "data_digest": digest}
    metadata_path = path.with_suffix(".json")
    if path.exists() or metadata_path.exists():
        if not path.is_file() or not metadata_path.is_file():
            raise FileNotFoundError(f"Incomplete dataset bundle at {path}.")
        recorded = json.loads(metadata_path.read_text(encoding="utf-8"))
        with np.load(path, allow_pickle=False) as stored:
            saved = {name: np.asarray(stored[name]) for name in stored.files}
        arrays_match = set(saved) == set(arrays) and all(
            np.array_equal(saved[name], value)
            for name, value in arrays.items()
        )
        if not arrays_match or recorded != metadata:
            raise ValueError(f"Existing bundle does not match request: {path}.")
        return False

    with tempfile.NamedTemporaryFile(
        mode="wb", dir=path.parent, prefix=f".{path.name}.", delete=False
    ) as handle:
        np.savez_compressed(handle, **arrays)
        temporary_data = Path(handle.name)
    os.replace(temporary_data, path)

    with tempfile.NamedTemporaryFile(
        mode="w",
        encoding="utf-8",
        dir=path.parent,
        prefix=f".{metadata_path.name}.",
        delete=False,
    ) as handle:
        json.dump(metadata, handle, indent=2, sort_keys=True)
        handle.write("\n")
        temporary_metadata = Path(handle.name)
    os.replace(temporary_metadata, metadata_path)
    return True


def file_digest(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def download_verified(url, path, expected_sha256):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.is_file():
        observed = file_digest(path)
        if observed != expected_sha256:
            raise ValueError(
                f"Archive digest mismatch for {path}: {observed}."
            )
        return False

    with tempfile.NamedTemporaryFile(
        mode="wb",
        dir=path.parent,
        prefix=f".{path.name}.",
        delete=False,
    ) as handle:
        temporary = Path(handle.name)
        with urllib.request.urlopen(url, timeout=120) as response:
            while chunk := response.read(1024 * 1024):
                handle.write(chunk)
    try:
        observed = file_digest(temporary)
        if observed != expected_sha256:
            raise ValueError(
                f"Downloaded archive digest mismatch: {observed}."
            )
        os.replace(temporary, path)
    finally:
        if temporary.exists():
            temporary.unlink()
    return True

In [ ]:
from fastCGC.experiments.upstream import verify_installed_distribution

DATASET_KEY = "causalChamber"
DATASET_DIR = (
    PROJECT_ROOT / "data" / "shared" / "causal-comparison"
    / DATASET_KEY / "datasets"
)
PACKAGE_REVISION = verify_installed_distribution(
    "causalchamber",
    expected_version="0.2.8",
)
from causalchamber.ground_truth import graph
ARCHIVE_PATH = Path(
    os.environ.get(
        "CAUSAL_CHAMBER_ARCHIVE",
        PROJECT_ROOT / "external" / "datasets" / "wt_walks_v1.zip",
    )
).expanduser()
download_verified(
    "https://causalchamber.s3.eu-central-1.amazonaws.com/downloadables/wt_walks_v1.zip",
    ARCHIVE_PATH,
    "1ef92d39fff7645a9e6e8cd41311e708362606491590a1534c6a5135d207736e",
)

VARIABLES = (
    "hatch", "load_in", "load_out", "pot_1", "pot_2",
    "current_in", "current_out", "pressure_downwind",
    "pressure_upwind", "rpm_in", "rpm_out", "mic",
    "pressure_intake", "pressure_ambient", "signal_1", "signal_2",
)
EXPERIMENTS = ("actuators_random_walk_1",)
FIT_ORDER = 10
MAX_NEW_DATASETS: int | None = None

reference_frame = graph("wt", "standard")
reference_graph = reference_frame.loc[list(VARIABLES), list(VARIABLES)].to_numpy(
    dtype=bool
)

created = 0
with zipfile.ZipFile(ARCHIVE_PATH) as archive:
    for experiment in EXPERIMENTS:
        if MAX_NEW_DATASETS is not None and created >= MAX_NEW_DATASETS:
            break
        member = f"wt_walks_v1/{experiment}.csv"
        if member not in archive.namelist():
            raise FileNotFoundError(f"{member} is absent from {ARCHIVE_PATH}.")
        with archive.open(member) as handle:
            frame = pd.read_csv(handle, usecols=list(VARIABLES))
        minima = frame.min(axis=0)
        spans = frame.max(axis=0) - minima
        if (spans <= 0).any():
            constant = spans.index[spans <= 0].tolist()
            raise ValueError(f"Constant wind-tunnel variables: {constant}.")
        normalized = (frame - minima) / spans
        series = normalized.to_numpy(dtype=float).T
        dataset_id = f"causal-chamber-wt-{experiment.replace('_', '-')}"
        was_created = save_bundle(
            DATASET_DIR / f"{dataset_id}.npz",
            series=series,
            reference_graph=reference_graph,
            metadata={
                "dataset_id": dataset_id,
                "dataset": DATASET_KEY,
                "data_source": "https://causalchamber.s3.eu-central-1.amazonaws.com/downloadables/wt_walks_v1.zip",
                "data_archive_sha256": "1ef92d39fff7645a9e6e8cd41311e708362606491590a1534c6a5135d207736e",
                "data_license": "CC BY 4.0",
                "ground_truth_package_source": "https://github.com/juangamella/causal-chamber-package.git",
                "upstream_version": PACKAGE_REVISION,
                "experiment": experiment,
                "fit_order": FIT_ORDER,
                "regime": "wind-tunnel-actuator-random-walk",
                "variable_names": list(VARIABLES),
                "n_variables": int(series.shape[0]),
                "n_samples": int(series.shape[1]),
                "preprocessing": "per-variable min-max normalization",
                "series_layout": "variable, time",
                "reference_layout": "source, target",
                "reference_scope": (
                    "physical causal model supplied by the Causal Chambers "
                    "authors; used as a contextual graph, not as a mandatory "
                    "lag-resolved scoring target"
                ),
                "evaluation": "qualitative inferred-graph comparison",
            },
        )
        created += int(was_created)
print(
    f"Created {created} new Causal Chambers bundles. "
    "Rerun to verify existing bundles and resume."
)